In [8]:
# Library Imports.
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn import metrics

# Allows plots to appear directly in the notebook.
%matplotlib inline


In [9]:
# Reading from a csv file, into a data frame
df_train = pd.read_csv('ppr-group-25204989-train-Formodeling.csv', keep_default_na=True, delimiter=',', skipinitialspace=True)
# Show data frame first few rows
df_train.head(10)

,Price(€),NotFullMarketPrice,VATExclusive,DescriptionofProperty,log_price,SaleYear,SaleMonthIndex,Region_Encoded
0,62500.0,0,0,0,11.042938,2016,7,11.664168
1,150000.0,1,0,0,11.918397,2016,12,12.000151
2,94000.0,0,0,0,11.451061,2016,8,11.664168
3,149500.0,0,1,1,11.915058,2016,2,11.056811
4,170000.0,0,0,0,12.043560,2016,10,11.290023
5,50000.0,0,0,0,10.819798,2016,12,11.821056
6,160000.0,0,0,0,11.982935,2016,12,12.288558
7,15000.0,0,0,0,9.615872,2016,10,13.078096
8,160000.0,0,0,0,11.982935,2016,8,11.642599
9,320000.0,0,1,1,12.676079,2016,12,11.886321


In [10]:
# Reading from a csv file, into a data frame
df_test = pd.read_csv('ppr-group-25204989-test-Forevaluation.csv', keep_default_na=True, delimiter=',', skipinitialspace=True)
# Show data frame first few rows
df_test.head(10)

,Price(€),NotFullMarketPrice,VATExclusive,DescriptionofProperty,log_price,SaleYear,SaleMonthIndex,Region_Encoded
0,2525000.00,0,0,0,14.741752,2025,120,13.352041
1,260000.00,0,0,0,12.468441,2025,118,12.181169
2,361233.00,0,1,1,12.797281,2025,120,12.536181
3,435000.00,0,0,0,12.983104,2025,114,12.882032
4,334000.00,0,0,0,12.718899,2025,119,12.447498
5,390000.00,0,0,0,12.873905,2025,119,12.734206
6,845814.97,0,1,1,13.648057,2025,117,13.055814
7,282000.00,0,0,0,12.549666,2025,109,12.368304
8,550000.00,0,0,0,13.217675,2025,112,12.562328
9,400000.00,0,0,0,12.899222,2025,118,12.616265


In [11]:
# This function is used repeatedly to compute, print and return the regression metrics
def getMetrics(testActualVal, predictions):
    mae=metrics.mean_absolute_error(testActualVal, predictions)
    rmse=metrics.mean_squared_error(testActualVal, predictions)**0.5
    r2=metrics.r2_score(testActualVal, predictions)
    print('\n==============================================================================')
    print("MAE: ", mae)
    #print("MSE: ", metrics.mean_squared_error(testActualVal, predictions))
    print("RMSE: ", rmse)
    print("R2: ", r2)
    return (mae,rmse,r2)

In [12]:
# Define descriptive features and target feature
features = ['NotFullMarketPrice', 'VATExclusive', 'SaleMonthIndex', 'Region_Encoded']
target_original = 'Price(€)'
target_log='log_price'

In [13]:
X_train = df_train[features]
y_train_log = df_train[target_log]

X_test = df_test[features]
y_test_original = df_test[target_original]


In [14]:
from sklearn.linear_model import RidgeCV


print("Ridge")

# train the model
ridge = RidgeCV()
ridge.fit(X_train, y_train_log)

# Print the weights learned for each feature.
print("\nIntercept: \n", ridge.intercept_)
print("Features and coeficients:", list(zip(features, ridge.coef_))[:10])

# Predicted price on validation set
pred_log = ridge.predict(X_test)
pred_original=np.expm1(pred_log)
(mae,rmse,r2)=getMetrics(y_test_original, pred_original)

feature_importance = pd.DataFrame({'feature': features, 'importance':ridge.coef_})
display(feature_importance.sort_values('importance', ascending=False))

Ridge

Intercept: 
 0.9836219748701609
Features and coeficients: [('NotFullMarketPrice', np.float64(-0.5128775035672204)), ('VATExclusive', np.float64(0.1836690096895115)), ('SaleMonthIndex', np.float64(0.0007240762724904926)), ('Region_Encoded', np.float64(0.9215391739580276))]

MAE:  185178.2201757136
RMSE:  1288761.6041125807
R2:  0.009851948045880299


,feature,importance
3,Region_Encoded,0.921539
1,VATExclusive,0.183669
2,SaleMonthIndex,0.000724
0,NotFullMarketPrice,-0.512878
